In [ ]:

using ITensors
using ITensorMPS
using JLD2
using Plots
using LinearAlgebra
using Statistics


theme(:wong)

data_file = "noise_vs_clean_data.jld2"


In [ ]:

"""
    load_data(filename::String)

Loads the simulation results from the JLD2 file.
Returns: (N_sorted, results_clean, results_noise)
"""
function load_data(filename::String)
    if !isfile(filename)
        error("File $filename not found.")
    end
    
    f = jldopen(filename, "r")
    results_clean = read(f, "results_clean")
    results_noise = read(f, "results_noise")
    close(f)
    
    # Sort keys to ensure plotting order is correct
    N_keys = intersect(keys(results_clean), keys(results_noise))
    N_sorted = sort(collect(N_keys))
    
    return N_sorted, results_clean, results_noise
end




In [ ]:

"""
    calc_entropy(psi::MPS, b::Int; alpha=1.0)

Calculates the entanglement entropy across bond `b`.
If alpha=1.0, returns Von Neumann entropy.
If alpha != 1.0, returns Renyi entropy of order alpha.
"""
function calc_entropy(psi::MPS, b::Int; alpha=1.0)
    # Orthogonalize to the bond to ensure valid Schmidt decomposition
    psi_c = copy(psi)
    orthogonalize!(psi_c, b)
    
    # SVD to get singular values
    # Left indices are everything to the left of bond b
    U, S, V = svd(psi_c[b], (linkind(psi_c, b-1), siteind(psi_c, b)))
    
    # Extract singular values (Schimidt coefficients)
    # S is a diagonal tensor
    sv = zeros(Float64, dim(S, 1))
    for i in 1:dim(S, 1)
        sv[i] = S[i, i]
    end
    
    # Normalize (just in case)
    p = sv.^2
    p = p ./ sum(p)
    
    # Remove zeros to avoid log errors
    p = p[p .> 1e-16]
    
    if alpha ≈ 1.0
        return -sum(p .* log.(p))
    else
        return (1.0 / (1.0 - alpha)) * log(sum(p.^alpha))
    end
end




In [ ]:

# ----------------------------------------------------------------------
# 1. Plot Bond Dimensions
# ----------------------------------------------------------------------
function plot_bond_dimensions(filename::String)
    N_list, res_clean, res_noise = load_data(filename)
    
    bd_clean = [maxlinkdim(res_clean[n]) for n in N_list]
    bd_noise = [maxlinkdim(res_noise[n]) for n in N_list]
    
    p = plot(N_list, bd_clean, label="Clean System", marker=:circle, linewidth=2,
             xlabel="System Size (N)", ylabel="Max Bond Dimension (χ)",
             title="Bond Dimension Scaling", legend=:topleft)
    
    plot!(p, N_list, bd_noise, label="Noise System", marker=:square, linewidth=2)
    
    display(p)
end


plot_bond_dimensions(data_file)
# Expectation: 
# Clean system -> Saturates (Area law) or grows slowly (Log law).
# Noise system -> Grows exponentially until hitting the max_bond_dim limit.


In [ ]:

# ----------------------------------------------------------------------
# 2. Plot Von Neumann Entropies
# ----------------------------------------------------------------------
function plot_von_neumann(filename::String)
    N_list, res_clean, res_noise = load_data(filename)
    
    vn_clean = Float64[]
    vn_noise = Float64[]
    
    for n in N_list
        # Cut at the middle of the chain
        b = n ÷ 2
        push!(vn_clean, calc_entropy(res_clean[n], b; alpha=1.0))
        push!(vn_noise, calc_entropy(res_noise[n], b; alpha=1.0))
    end
    
    p = plot(N_list, vn_clean, label="Clean (VN)", marker=:circle, linewidth=2,
             xlabel="System Size (N)", ylabel="Entanglement Entropy S_vn",
             title="Von Neumann Entropy (Half-Chain)", legend=:topleft)
             
    plot!(p, N_list, vn_noise, label="Noise (VN)", marker=:square, linewidth=2)
    
    display(p)
end



plot_von_neumann(data_file)

In [ ]:

# ----------------------------------------------------------------------
# 3. Plot Renyi Entropies
# ----------------------------------------------------------------------
function plot_renyi_entropy(filename::String; alpha=2.0)
    N_list, res_clean, res_noise = load_data(filename)
    
    r_clean = Float64[]
    r_noise = Float64[]
    
    for n in N_list
        b = n ÷ 2
        push!(r_clean, calc_entropy(res_clean[n], b; alpha=alpha))
        push!(r_noise, calc_entropy(res_noise[n], b; alpha=alpha))
    end
    
    p = plot(N_list, r_clean, label="Clean (Renyi α=$alpha)", marker=:circle, linewidth=2,
             xlabel="System Size (N)", ylabel="Renyi Entropy S_$alpha",
             title="Renyi Entropy (Half-Chain)", legend=:topleft)
             
    plot!(p, N_list, r_noise, label="Noise (Renyi α=$alpha)", marker=:square, linewidth=2)
    
    display(p)
end




plot_renyi_entropy(data_file, alpha=2.0)

In [ ]:

# ----------------------------------------------------------------------
# 4. Plot Schmidt Spectrum
# ----------------------------------------------------------------------
function plot_schmidt_spectrum(filename::String, N_target::Int)
    _, res_clean, res_noise = load_data(filename)
    
    if !haskey(res_clean, N_target)
        println("Data for N=$N_target not found.")
        return
    end
    
    # Get states
    psi_c = res_clean[N_target]
    psi_n = res_noise[N_target]
    b = N_target ÷ 2
    
    # Helper to get spectrum
    function get_spectrum(psi, b)
        psi_copy = copy(psi)
        orthogonalize!(psi_copy, b)
        U, S, V = svd(psi_copy[b], (linkind(psi_copy, b-1), siteind(psi_copy, b)))
        return [S[i,i] for i in 1:dim(S,1)]
    end
    
    spec_clean = get_spectrum(psi_c, b)
    spec_noise = get_spectrum(psi_n, b)
    
    # Plotting log scale
    p = plot(1:length(spec_clean), log10.(spec_clean.^2), label="Clean", 
             xlabel="Index k", ylabel="log10(Schmidt Coeff^2)",
             title="Schmidt Spectrum (N=$N_target)", linewidth=2)
             
    plot!(p, 1:length(spec_noise), log10.(spec_noise.^2), label="Noise", linewidth=2)
    
    display(p)
end


plot_schmidt_spectrum(data_file, 30)
# Expectation:
# Clean -> Decays very fast (few dominant states).
# Noise -> Decays slowly (many states contribute), characteristic of complex/chaotic systems.



In [ ]:

# ----------------------------------------------------------------------
# 5. SYK / Volume Law Comparison
# ----------------------------------------------------------------------
function plot_noise_vs_syk_model(filename::String)
    N_list, _, res_noise = load_data(filename)
    
    # 1. Get Noise Data (Bond Dimension)
    bd_noise = [maxlinkdim(res_noise[n]) for n in N_list]
    
    # 2. Create Theoretical Comparisons
    # The SYK model and random unitary circuits typically exhibit Volume Law entanglement.
    # For Entropy: S ~ N/2 * log(2)
    # For Bond Dimension: χ ~ exp(S) ~ exp(N)
    # We fit a simple exponential curve to the noise data to show the trend
    
    # Theoretical Volume Law Scaling (Generic for SYK/Random systems)
    # We normalize it to start near the data for visual comparison
    # S_vol ~ N (Volume Law) -> Chi_vol ~ 2^(N/constant)
    
    # Simple exponential ansatz: A * e^(B*N)
    # Using log-linear regression for rough fit
    Y = log.(bd_noise)
    X = N_list
    # Linear fit Y = a + bX
    mx = mean(X); my = mean(Y)
    b_slope = sum((X .- mx) .* (Y .- my)) / sum((X .- mx).^2)
    a_intercept = my - b_slope * mx
    
    syk_proxy = exp.(a_intercept .+ b_slope .* N_list)
    
    # Plot
    p = plot(N_list, bd_noise, label="Noise System Data", 
             marker=:square, linewidth=2, color=:red,
             xlabel="System Size (N)", ylabel="Bond Dimension (χ)",
             title="Noise System vs Volume Law (SYK-like) Scaling",
             legend=:topleft, yscale=:log10)
             
    plot!(p, N_list, syk_proxy, label="Exponential Fit (~ e^$(round(b_slope, digits=2))N)", 
          linestyle=:dash, color=:black, linewidth=2)

    # Add text annotation
    annotate!(p, N_list[end-5], bd_noise[end], text("Volume Law", :bottom, 10))
    
    display(p)
end


plot_noise_vs_syk_model(data_file)
# This plots the Noise data on a Log-Y scale. 
# A straight line on Log-Y indicates Exponential growth (Volume Law), 
# which matches SYK/Random Unitary behavior.



In [ ]:

# ----------------------------------------------------------------------
# 6. Extra: Spin Correlations (Glassy vs Ordered)
# ----------------------------------------------------------------------
"""
Calculates correlations <Sz_1 Sz_r> to distinguish between 
Néel order (Clean) and Spin Glass/Disorder (Noise).
"""
function plot_spin_correlations(filename::String, N_target::Int)
    _, res_clean, res_noise = load_data(filename)
    
    if !haskey(res_clean, N_target)
        println("N=$N_target not found.")
        return
    end

    psi_c = res_clean[N_target]
    psi_n = res_noise[N_target]

    # Calculate correlation from site 1 to all other sites
    # Using ITensors `correlation_matrix` is usually fastest, 
    # but let's do simple 1-to-r measurement
    
    corrs_c = correlation_matrix(psi_c, "Sz", "Sz")[1, :]
    corrs_n = correlation_matrix(psi_n, "Sz", "Sz")[1, :]
    
    # Take absolute value for Noise to see "Glassy" strength vs "Néel" oscillation
    # Or plot raw to see the staggering
    
    r_vals = 1:N_target
    
    p1 = plot(r_vals, real(corrs_c), title="Clean Correlations <Sz_1 Sz_r>", 
              xlabel="r", ylabel="Corr", marker=:circle, label="Clean")
              
    p2 = plot(r_vals, real(corrs_n), title="Noise Correlations <Sz_1 Sz_r>", 
              xlabel="r", ylabel="Corr", marker=:circle, color=:red, label="Noise")
              
    display(plot(p1, p2, layout=(2,1)))
end

plot_spin_correlations(data_file, 30)
# Clean -> Likely oscillating (+ - + -) for Neel.
# Noise -> Random fluctuations, but likely short range if fully thermalized, or random long range.